# 🛡️ Mini SOC — GRPO Training Notebook

Train an RL agent to operate a Security Operations Center using **Group Relative Policy Optimization (GRPO)**.

| Component | Details |
|---|---|
| **Base Model** | Qwen/Qwen2.5-1.5B-Instruct |
| **Method** | GRPO + LoRA (4.3M params) |
| **Framework** | HuggingFace TRL |
| **GPU** | T4 (16GB VRAM) |
| **Training Time** | ~2h for 200 steps |
| **Environment** | [HF Space](https://huggingface.co/spaces/riteshp30/mini-soc) |

**Cells:**
1. Install dependencies & clone repo
2. Connect to Mini SOC environment
3. Run random baseline ("before" evidence)
4. GRPO training (200 steps)
5. View TensorBoard logs
6. Evaluate trained model ("after" evidence)
7. Push to HF Hub (optional)

> **GPU Required:** Runtime → Change runtime type → GPU → T4

In [ ]:
# Cell 1: Install dependencies & clone repo
!pip install -q \
    "trl>=0.15.0" \
    "peft>=0.14.0" \
    "transformers>=4.48.0" \
    datasets \
    httpx \
    torch \
    matplotlib \
    tensorboard \
    accelerate

# Force-reinstall TRL to avoid version conflicts
!pip install -U --force-reinstall --no-deps "trl>=0.15.0,<1.0" -q

# Clone the Mini SOC repository
import os
os.chdir('/content')
!rm -rf mini-soc
!git clone https://github.com/riteshthekid/mini-soc.git
os.chdir('/content/mini-soc')

# Verify GPU
import torch
print(f'\n✅ Dependencies installed')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 2: Connect to Mini SOC environment
import os, time, httpx

SOC_ENV_URL = 'https://riteshp30-mini-soc.hf.space'
os.environ['SOC_ENV_URL'] = SOC_ENV_URL
os.environ['WANDB_DISABLED'] = 'true'

print(f'🌐 Using HF Space: {SOC_ENV_URL}')

# Wake the Space if sleeping
for attempt in range(5):
    try:
        r = httpx.get(f'{SOC_ENV_URL}/health', timeout=60)
        if r.status_code == 200:
            print(f'✅ Space is awake: {r.json()}')
            break
    except Exception as e:
        print(f'   Waiting for Space to wake... ({attempt+1}/5): {e}')
        time.sleep(15)
else:
    raise RuntimeError('HF Space unreachable')

# Sanity check all 3 tasks
print('\n📋 Testing all 3 tasks:')
for task in ['alert_triage', 'incident_investigation', 'threat_response']:
    r = httpx.post(f'{SOC_ENV_URL}/reset', json={'task_id': task}, timeout=30)
    obs = r.json()['observation']
    print(f'   {task}: alerts={len(obs["alert_queue"])}, logs={len(obs["available_logs"])}')

print('\n✅ Environment ready for training!')

In [ ]:
# Cell 3: Random Baseline ("Before" evidence)
# ⚠️ Screenshot this output!
import httpx, random

TASK_IDS = ['alert_triage', 'incident_investigation', 'threat_response']

def run_random_agent(task_id, num_episodes=3):
    scores = []
    actions = ['query_logs', 'classify_alert', 'isolate_asset', 'block_ip', 'request_info']
    for ep in range(num_episodes):
        httpx.post(f'{SOC_ENV_URL}/reset', json={'task_id': task_id}, timeout=30)
        done, final_score, steps = False, 0.0, 0
        while not done and steps < 20:
            action = random.choice(actions)
            params = {}
            if action == 'query_logs':
                params = {'log_source': random.choice(['auth', 'firewall', 'dns', 'process', 'network'])}
            elif action == 'classify_alert':
                params = {'alert_id': f'ALT-{random.randint(1,49):03d}',
                          'classification': random.choice(['benign','suspicious','critical']),
                          'priority': random.choice(['P1','P2','P3','P4'])}
            elif action == 'isolate_asset':
                params = {'hostname': random.choice(['WEB-SERVER-01', 'DC-01', 'WS-HR-03'])}
            elif action == 'block_ip':
                params = {'ip_address': f'{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}'}
            try:
                result = httpx.post(f'{SOC_ENV_URL}/step',
                    json={'action_type': action, 'parameters': params}, timeout=30).json()
                done = result.get('done', False)
                if done: final_score = result.get('info', {}).get('final_score', 0.0)
                steps += 1
            except: break
        scores.append(final_score)
    return sum(scores) / len(scores) if scores else 0.0

print('Running random agent baseline (3 episodes each)...')
baseline_scores = {}
for task in TASK_IDS:
    score = run_random_agent(task)
    baseline_scores[task] = round(score, 4)
    print(f'   {task:<30} {score:.4f}')

avg = sum(baseline_scores.values()) / len(baseline_scores)
print(f'\n   {"Average":<30} {avg:.4f}')
print(f'\n📸 Screenshot this — it is your BEFORE evidence for judges.')

In [ ]:
# Cell 4: GRPO Training (200 steps, ~2h on T4)
import os, sys, logging
os.environ.setdefault('SOC_ENV_URL', 'https://riteshp30-mini-soc.hf.space')
os.environ['WANDB_DISABLED'] = 'true'
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(name)-20s | %(message)s')
os.chdir('/content/mini-soc')
if '/content/mini-soc' not in sys.path:
    sys.path.insert(0, '/content/mini-soc')

from train.train_grpo import run_training

output_dir = run_training(
    num_steps=200,
    batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_generations=4,
    max_completion_length=200,
    temperature=0.7,
    num_prompts=60,
    save_steps=50,
    logging_steps=5,
    push_to_hub=False,
    use_wandb=False,
    use_unsloth=False,
    log_file='./outputs/mini-soc-grpo/training_log.jsonl',
)
print(f'\n🎉 Training complete! Model saved to: {output_dir}')

In [ ]:
# Cell 5: TensorBoard — View training metrics
%load_ext tensorboard
%tensorboard --logdir ./outputs/mini-soc-grpo/runs/

In [ ]:
# Cell 6: Evaluate Trained Model + Before/After Comparison
# ⚠️ Screenshot this output!
import os, sys, json, re, httpx, torch
os.chdir('/content/mini-soc')
if '/content/mini-soc' not in sys.path:
    sys.path.insert(0, '/content/mini-soc')

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from train.plot_rewards import plot_reward_curve, plot_comparison

MODEL_PATH = './outputs/mini-soc-grpo'
ENV = os.environ.get('SOC_ENV_URL', 'https://riteshp30-mini-soc.hf.space')

# Hardcode baselines if runtime restarted
if 'baseline_scores' not in dir():
    baseline_scores = {'alert_triage': 0.001, 'incident_investigation': 0.001, 'threat_response': 0.0}

# Load trained model (base + LoRA adapter)
print('Loading trained model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
base_model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-1.5B-Instruct', torch_dtype=torch.float16, device_map='auto')
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model = model.merge_and_unload()
model.eval()
print('Model loaded\n')

SYSTEM_PROMPT = (
    'You are a SOC analyst AI. You operate a security environment by issuing '
    'JSON actions. Each action must be a JSON object with "action_type" and '
    '"parameters" keys.\n\n'
    'Available action_types: query_logs, classify_alert, isolate_asset, '
    'block_ip, escalate, write_report, close_incident, request_info.\n\n'
    'Respond with ONLY valid JSON. No explanations.\n'
    'Example: {"action_type": "classify_alert", "parameters": '
    '{"alert_id": "ALT-001", "classification": "critical", "priority": "P1"}}'
)

def extract_json(text):
    try: return json.loads(text.strip())
    except: pass
    m = re.search(r'\{[^{}]*\}', text)
    if m:
        try: return json.loads(m.group())
        except: pass
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    return None

print('Evaluating trained model...')
trained_scores = {}
for task_id in ['alert_triage', 'incident_investigation', 'threat_response']:
    result = httpx.post(f'{ENV}/reset', json={'task_id': task_id}, timeout=30).json()
    json_ok = 0
    for step in range(20):
        obs_text = json.dumps(result.get('observation', {}), indent=2)[:1500]
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': f'Task: {task_id}\nObservation:\n{obs_text}\nRespond with ONE JSON action:'},
        ]
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text_input, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=150, temperature=0.3, do_sample=True)
        text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        action = extract_json(text)
        if action and 'action_type' in action:
            json_ok += 1
        else:
            action = {'action_type': 'request_info', 'parameters': {}}
        try:
            result = httpx.post(f'{ENV}/step', json=action, timeout=15).json()
            if result.get('done'): break
        except: break
    score = result.get('info', {}).get('final_score', 0.0)
    trained_scores[task_id] = round(score, 4)
    print(f'  {task_id:<30} {score:.4f}  (JSON: {json_ok}/{step+1})')

# Print comparison table
print(f'\n{"="*70}')
print(f'{"Task":<30} {"Baseline":>15} {"Trained":>15} {"Delta":>10}')
print(f'{"-"*70}')
for task in trained_scores:
    b = baseline_scores.get(task, 0.0)
    a = trained_scores[task]
    print(f'{task:<30} {b:>15.4f} {a:>15.4f} {a-b:>+10.4f}')
avg_b = sum(baseline_scores.values()) / 3
avg_a = sum(trained_scores.values()) / 3
print(f'{"-"*70}')
print(f'{"AVERAGE":<30} {avg_b:>15.4f} {avg_a:>15.4f} {avg_a-avg_b:>+10.4f}')
print(f'{"="*70}')

# Generate charts
try:
    plot_reward_curve('./outputs/mini-soc-grpo/training_log.jsonl',
                      './outputs/reward_curve.png', show_plot=True)
except: print('(No training log found for reward curve)')

plot_comparison(baseline_scores, trained_scores,
                './outputs/comparison_chart.png', show_plot=True)
print('\n📸 Screenshot the charts and table above for your submission!')

In [ ]:
# Cell 7: Push Trained Model to HuggingFace Hub (Optional)
# Uncomment and set your HF token to push the model

# from huggingface_hub import HfApi, login
# login()  # Will prompt for your HF token
# api = HfApi()
# api.upload_folder(
#     folder_path='./outputs/mini-soc-grpo',
#     repo_id='riteshp30/mini-soc-grpo-v1',
#     repo_type='model',
# )
# print('✅ Model pushed to HF Hub: riteshp30/mini-soc-grpo-v1')